# 07 Learning Rate and Early Stopping

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`exp_cmb008_s4_learning_rate.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s4_learning_rate.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb008_s5_early_stopping.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_early_stopping.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb008_s4_s5_aggregate.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s4_s5_aggregate.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb008_s4_s5_finalize.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s4_s5_finalize.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/MLP/Combined/scripts/exp_cmb008_s4_learning_rate.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'ad04c75614ce8127309fe5d33e10019bbdd37044'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_early_stopping.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '92bef6cdfed15d74a2bb2441814ce00108bbdc3b'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb008_s4_s5_aggregate.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '1a6a6772c4ada202b531b80c8dc3d16b97243ea2'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb008_s4_s5_finalize.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'c8a2040fc323c040ff3963607edf9d74accf234e'}}

## exp_cmb008_s4_learning_rate

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb008_s4_learning_rate.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s4_learning_rate.py)  
Git blob: `ad04c75614ce8127309fe5d33e10019bbdd37044`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-008 / S4. Freeze (256,), alpha=1e-4; compare Adam learning_rate_init.

S4 learning rates 0.0005 and 0.002 use the original S1 preprocessor and writer folds;
S1 (256,), alpha=1e-4, LR=0.001, ES=False is archived baseline.
3 seeds (42/43/44) and 4 writer-held-out folds per candidate.
No Original/Professor Test or new 480. No held-out/OfficialVal training augmentation.

CLI 인자 (정적 추출): `--arm`, `--learning-rate`, `--seed`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb008_s4_learning_rate.py
#!/usr/bin/env python3
"""EXP-CMB-008 / S4. Freeze (256,), alpha=1e-4; compare Adam learning_rate_init.

S4 learning rates 0.0005 and 0.002 use the original S1 preprocessor and writer folds;
S1 (256,), alpha=1e-4, LR=0.001, ES=False is archived baseline.
3 seeds (42/43/44) and 4 writer-held-out folds per candidate.
No Original/Professor Test or new 480. No held-out/OfficialVal training augmentation.
"""
from __future__ import annotations
import argparse,csv,hashlib,json,sys,time,warnings
from pathlib import Path
import joblib
import numpy as np
import sklearn
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score,balanced_accuracy_score,roc_auc_score,
                             precision_recall_fscore_support,confusion_matrix)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT=Path(__file__).resolve().parents[4]
CMB=ROOT/"05_model_optimize"/"MLP"/"Combined"
NORMAL=ROOT/"05_model_optimize"/"MLP"/"Normal"
sys.path.insert(0,str(CMB/"scripts"))
sys.path.insert(0,str(NORMAL/"scripts"))
from combined_mlp_writer_cv import (load_official,load_writers,PixelScaler255,
                                    WRITERS,LABELS,V3_SHA,HANDMADE_SHA)
from exp006_size_com_comparison import process,image_geometry
from exp015_scale_stroke_grid import nested_subset
from exp016_high_fraction_grid import scale_base,stroke_jitter
from exp014_balanced_augmentation import balanced_indices
from exp011_012_causal_probes import fixed_scale

SCALES=(.8,.9,1.,1.1,1.2)
PAIRS=(("0","D"),("D","0"),("8","B"),("B","8"),("5","F"),("F","5"),
       ("7","9"),("7","A"),("4","F"))
ARMS={"com_only":(0.,0.),
      "scale10_stroke00":(.1,0.),
      "scale10_stroke14":(.1,.14),
      "scale10_stroke28":(.1,.28),
      "scale20_stroke00":(.2,0.),
      "scale20_stroke14":(.2,.14),
      "scale20_stroke28":(.2,.28),
      "scale30_stroke00":(.3,0.),
      "scale30_stroke14":(.3,.14),
      "scale30_stroke28":(.3,.28)}
TARGETS=(16,18,20,22,24)
SEEDS=(42,43,44)
CLASSES=np.arange(16,dtype=int)

def savecsv(p,records):
    records=list(records)
    if not records:raise RuntimeError("missing rows "+str(p))
    p.parent.mkdir(parents=True,exist_ok=True)
    with p.open("w",newline="",encoding="utf-8-sig") as f:
        wr=csv.DictWriter(f,fieldnames=list(records[0]))
        wr.writeheader();wr.writerows(records)

def metrics(y,p,subset=None):
    labels=CLASSES if subset is None else np.asarray(subset)
    prec,rec,f1,sup=precision_recall_fscore_support(y,p,labels=labels,zero_division=0)
    return {"n":int(len(y)),"correct":int(np.sum(p==y)),
            "accuracy":float(np.mean(p==y)),
            "macro_f1":float(f1.mean()),
            "macro_precision":float(prec.mean()),
            "macro_recall":float(rec.mean())}

def confusion_pairs(y,p,writer,arm,seed,source):
    """All 240 off-diagonal directions, including zero-error pairs.

    Denominator n_actual corresponds to the *actual* (row) class.
    Both counts and conditional percentages are saved; total overall
    accuracy does not replace per-class and per-pair error analysis.
    """
    cm=confusion_matrix(y,p,labels=CLASSES)
    result=[]
    nall=int(len(y))
    nwrong=int(nall-np.trace(cm))
    for actual in CLASSES:
        nactual=int(cm[actual].sum())
        wrong=int(nactual-cm[actual,actual])
        for target in CLASSES:
            if actual==target:continue
            ct=int(cm[actual,target])
            result.append({"arm":arm,"seed":seed,"dataset":source,"writer":writer,
                 "actual":LABELS[actual],"predicted":LABELS[target],
                 "n_actual":nactual,"n_actual_wrong":wrong,
                 "n_dataset":nall,"n_dataset_wrong":nwrong,
                 "errors":ct,"error_rate_within_actual_class":ct/nactual if nactual else 0,
                 "share_of_actual_class_errors":ct/wrong if wrong else 0,
                 "share_of_all_dataset_errors":ct/nwrong if nwrong else 0})
    assert len(result)==16*15
    assert sum(int(r["errors"]) for r in result)==nwrong
    return result

def total_errors_by_class(y,p,writer,arm,seed,source):
    """All 16 true classes: total incorrect / n actual class, plus group switch."""
    cm=confusion_matrix(y,p,labels=CLASSES)
    out=[]
    for a in CLASSES:
        n=int(cm[a].sum());correct=int(cm[a,a]);wrong=n-correct
        family_swap=int(cm[a,10:].sum()) if a<10 else int(cm[a,:10].sum())
        out.append({"arm":arm,"seed":seed,"dataset":source,"writer":writer,
           "actual":LABELS[a],"n":n,"correct":correct,"wrong":wrong,
           "class_error_rate":wrong/n if n else 0,
           "opposite_family_errors":family_swap,
           "opposite_family_error_rate":family_swap/n if n else 0})
    assert len(out)==16
    return out

def family_error_breakdown(y,p,writer,arm,seed,source):
    """Digits→letters and letters→digits alongside within-family mistakes."""
    cm=confusion_matrix(y,p,labels=CLASSES)
    d2l=int(cm[:10,10:].sum());l2d=int(cm[10:,:10].sum())
    d2d=int(cm[:10,:10].sum()-np.trace(cm[:10,:10]))
    l2l=int(cm[10:,10:].sum()-np.trace(cm[10:,10:]))
    return [{"arm":arm,"seed":seed,"dataset":source,"writer":writer,
      "direction":key,"errors":val,"n_actual_family":n,
      "error_rate":val/n if n else 0}
      for key,val,n in (("digit_to_letter",d2l,int(cm[:10].sum())),
                        ("letter_to_digit",l2d,int(cm[10:].sum())),
                        ("digit_to_digit",d2d,int(cm[:10].sum())),
                        ("letter_to_letter",l2l,int(cm[10:].sum())))]

def geom_rows(name,x,y,arm,seed):
    geo=image_geometry(x)
    out=[]
    for nameg,mask in (("digits",y<10),("letters",y>=10)):
        z={"arm":arm,"seed":seed,"dataset":name,"group":nameg,
           "n":int(mask.sum())}
        for field in ("long_side","width","height","mass"):
            zz=np.asarray(geo[field])[mask]
            z[field+"_mean"]=float(zz.mean())
            z[field+"_std"]=float(zz.std())
        out.append(z)
    return out

def binary_size_probe(xtrain,ytrain,xval,yval):
    train=image_geometry(xtrain)["long_side"].reshape(-1,1)
    val=image_geometry(xval)["long_side"].reshape(-1,1)
    model=Pipeline([("std",StandardScaler()),
      ("logistic",LogisticRegression(class_weight="balanced",random_state=42,max_iter=1000))])
    model.fit(train,(ytrain>=10).astype(int))
    q=model.predict_proba(val)[:,1]
    return {"bal_acc":float(balanced_accuracy_score(yval>=10,q>=.5)),
            "auc":float(roc_auc_score(yval>=10,q))}

def prepareraw(official,writers,arm):
    ratio,stroke=ARMS[arm]
    raw=official["X_train"]
    yy=official["y_train"]
    # CoM-only is exactly the EXP-CMB-004 reference condition.
    main=process(raw,"com",20)
    changed={"method":arm,"requested_ratio":ratio,"actual_ratio":0.,
             "n_aug":0,"n_stroke_changed":0,"n_stroke_rejected":0,
             "official_train_rows":len(raw)}
    targets=[]
    if ratio:
        ids,target_map=nested_subset(yy,ratio)  # Normal EXP015 nested low-ratio group
        half,_=balanced_indices(yy,seed=20261116)
        augmented=scale_base(raw,ids,target_map)
        augmented,stroke_stats=stroke_jitter(augmented,stroke,ids,half)
        main[ids]=process(augmented,"com",20)
        changed.update({"n_aug":len(ids),"actual_ratio":len(ids)/len(yy),
             "n_stroke_changed":stroke_stats["modified"],
             "n_stroke_rejected":stroke_stats["rejected"]})
        for cl in CLASSES:
            for length in TARGETS:
                ct=int(np.sum((yy[ids]==cl)&(target_map[ids]==length)))
                targets.append({"arm":arm,"class":LABELS[cl],"target_size":length,
                                "train_images_assigned":ct})
                assert ct>=1
        for cl in CLASSES:
            cc=[x["train_images_assigned"] for x in targets if x["class"]==LABELS[cl]]
            assert max(cc)-min(cc)<=1
        assert abs(len(ids)/len(yy)-ratio)<.002
    else:
        for cl in CLASSES:
            for length in TARGETS:
                targets.append({"arm":arm,"class":LABELS[cl],"target_size":length,
                                "train_images_assigned":0})
    transformed_w={w:process(writers[w]["X"],"com",20) for w in WRITERS}
    assert main.shape==(76800,784) and len(transformed_w)==4
    return main,transformed_w,changed,targets

def make_scaled(official,writers):
    ref={}
    for scale in SCALES:
        # exact Normal EXP012 intervention; raw image BBox-centered affine, then CoM
        ref[("official",scale)]=process(fixed_scale(official["X_val"],scale),"com",20)
        for w in WRITERS:
            ref[(w,scale)]=process(fixed_scale(writers[w]["X"],scale),"com",20)
    return ref

def eval_scales(model,raw_map,y,dataset,writer,arm,seed):
    rows=[]; pair_rows=[]; focus=[]; class_errors=[]; family_errors=[]
    probs={}
    for sc in SCALES:
        xx=raw_map[(writer if dataset=="OOF" else "official",sc)]
        q=np.asarray(model.predict_proba(xx),dtype=np.float64)
        assert np.array_equal(model.named_steps["model"].classes_,CLASSES)
        p=q.argmax(axis=1).astype(int)
        letter=q[:,10:].sum(axis=1)
        probs[sc]=(p,letter)
    _,base_letter=probs[1.]
    base_family=base_letter>=.5
    for sc in SCALES:
        p,letter=probs[sc]
        for group,z,cls in (("all",np.ones(len(y),dtype=bool),CLASSES),
             ("digits",y<10,np.arange(10)),("letters",y>=10,np.arange(10,16))):
            m=metrics(y[z],p[z],cls)
            flip=(letter[z]>=.5)!=(base_family[z])
            rows.append({"arm":arm,"seed":seed,"writer":writer,"dataset":dataset,
               "scale":sc,"group":group,**m,
               "family_flip":float(flip.mean()),
               "mean_abs_delta_p_letter":float(np.abs(letter[z]-base_letter[z]).mean()),
               "mean_p_letter":float(letter[z].mean())})
        for cls_i in CLASSES:
            z=y==cls_i
            pz=p[z];ll=letter[z];bb=base_letter[z]
            focus.append({"arm":arm,"seed":seed,"writer":writer,"dataset":dataset,
                "scale":sc,"actual":LABELS[cls_i],"n":int(z.sum()),
                "correct":int(np.sum(pz==cls_i)),
                "accuracy":float(np.mean(pz==cls_i)),
                "family_flip":float(np.mean((ll>=.5)!=(bb>=.5))),
                "mean_abs_delta_p_letter":float(np.mean(np.abs(ll-bb))),
                "mean_p_letter":float(ll.mean())})
        sc_name=dataset+"_scale_"+str(sc)
        pair_rows+=confusion_pairs(y,p,writer,arm,seed,sc_name)
        class_errors+=total_errors_by_class(y,p,writer,arm,seed,sc_name)
        family_errors+=family_error_breakdown(y,p,writer,arm,seed,sc_name)
    return rows,pair_rows,focus,probs[1.][0],class_errors,family_errors

def output_plots(y,p,out,arm,seed):
    cm=confusion_matrix(y,p,labels=CLASSES)
    for k in ("counts","row_normalized"):
        v=cm if k=="counts" else 100*cm/np.maximum(cm.sum(axis=1,keepdims=True),1)
        fig,ax=plt.subplots(figsize=(10,8))
        im=ax.imshow(v,cmap="viridis",vmin=0,vmax=160 if k=="counts" else 100)
        fig.colorbar(im,ax=ax,fraction=.045,pad=.04)
        ax.set(xticks=np.arange(16),yticks=np.arange(16),
               xticklabels=list(LABELS),yticklabels=list(LABELS),
               xlabel="Predicted",ylabel="True",
               title=f"EXP-CMB-005 {arm} seed={seed} Combined OOF {k}")
        fig.tight_layout()
        fig.savefig(out/f"OOF_confusion_{k}.png",dpi=145)
        plt.close(fig)

def run(arm,seed,out,learning_rate):
    if arm!="scale30_stroke28" or seed not in SEEDS or learning_rate not in (.0005,.002):raise ValueError((arm,seed,learning_rate))
    if out.exists(): raise FileExistsError(out)
    out.mkdir(parents=True)
    (out/"models").mkdir()
    t0=time.perf_counter()
    original=ROOT/"02_data"/"datasets"/"MNIST_EMNIST_official_split_112000.npz"
    handmade=ROOT/"02_data"/"Hand_made"
    off=load_official(original)
    writers=load_writers(handmade)
    assert off["sha256"]==V3_SHA
    for w in WRITERS:assert writers[w]["sha256"]==HANDMADE_SHA[w]
    assert len(off["X_train"])==76800 and len(off["X_val"])==19200
    tpre=time.perf_counter()
    xtrain_off,xwriters,info,targets=prepareraw(off,writers,arm)
    xval=process(off["X_val"],"com",20)
    scale_map=make_scaled(off,writers)
    preprocess_time=time.perf_counter()-tpre
    official_probe=binary_size_probe(xtrain_off,off["y_train"],xval,off["y_val"])
    savecsv(out/"target_assignments.csv",targets)
    features=geom_rows("official_train",xtrain_off,off["y_train"],arm,seed)
    features+=geom_rows("official_val",xval,off["y_val"],arm,seed)
    for w in WRITERS:features+=geom_rows("writer_"+str(w),xwriters[w],writers[w]["y"],arm,seed)
    savecsv(out/"geometry_distributions.csv",features)
    foldrows=[];pairs=[];class_errors=[];family_errors=[];scale_rows=[];class_rows=[];scale_class_rows=[];pred_rows=[];loss=[];fit=[]
    pooled=[]
    for held in WRITERS:
        trwriters=[w for w in WRITERS if w!=held]
        X=np.concatenate([xtrain_off]+[xwriters[w] for w in trwriters])
        y=np.concatenate([off["y_train"]]+[writers[w]["y"] for w in trwriters])
        assert X.shape==(78720,784) and len(y)==78720
        mlp=Pipeline([("scale_0_1",PixelScaler255()),
                      ("model",MLPClassifier(random_state=seed,hidden_layer_sizes=(256,),alpha=0.0001,learning_rate_init=learning_rate))])
        started=time.perf_counter()
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            mlp.fit(X,y)
        sec=time.perf_counter()-started
        mlpnet=mlp.named_steps["model"]
        assert mlpnet.n_features_in_==784
        for it,lossval in enumerate(mlpnet.loss_curve_):
            loss.append({"arm":arm,"seed":seed,"heldout":held,
                        "iteration":it+1,"loss":float(lossval)})
        fit.append({"arm":arm,"seed":seed,"heldout":held,
                    "train_n":len(y),"heldout_n":640,
                    "fit_seconds":sec,"iterations":mlpnet.n_iter_,
                    "last_loss":float(mlpnet.loss_),
                    "warnings":" | ".join(str(w.message) for w in caught)})
        fout=out/"models"/f"heldout_{held}.joblib"
        joblib.dump(mlp,fout,compress=3)
        handwritten_y=writers[held]["y"]
        outcome=mlp.predict(xwriters[held]).astype(int)
        pp=mlp.predict(xval).astype(int)
        for ds,ytrue,pred in (("OOF",handwritten_y,outcome),
                              ("OfficialVal",off["y_val"],pp)):
            maskset=(("all",np.ones(len(ytrue),bool),CLASSES),
                     ("digits",ytrue<10,np.arange(10)),
                     ("letters",ytrue>=10,np.arange(10,16)))
            for g,z,classes in maskset:
                foldrows.append({"arm":arm,"seed":seed,"heldout":held,"dataset":ds,
                      "group":g,**metrics(ytrue[z],pred[z],classes)})
        prec,rec,f1,_=precision_recall_fscore_support(handwritten_y,outcome,
              labels=CLASSES,zero_division=0)
        for ci in CLASSES:
            z=handwritten_y==ci
            class_rows.append({"arm":arm,"seed":seed,"writer":held,
               "class":LABELS[ci],"n":int(z.sum()),"correct":int(np.sum(outcome[z]==ci)),
               "precision":float(prec[ci]),"recall":float(rec[ci]),"f1":float(f1[ci])})
        for i in range(640):
            pred_rows.append({"arm":arm,"seed":seed,"writer":held,"index":i,
                "actual":LABELS[int(handwritten_y[i])],
                "pred":LABELS[int(outcome[i])]})
        pooled.append(outcome)
        pairs+=confusion_pairs(handwritten_y,outcome,held,arm,seed,"OOF_original")
        pairs+=confusion_pairs(off["y_val"],pp,held,arm,seed,"OfficialVal_original")
        class_errors+=total_errors_by_class(handwritten_y,outcome,held,arm,seed,"OOF_original")
        class_errors+=total_errors_by_class(off["y_val"],pp,held,arm,seed,"OfficialVal_original")
        family_errors+=family_error_breakdown(handwritten_y,outcome,held,arm,seed,"OOF_original")
        family_errors+=family_error_breakdown(off["y_val"],pp,held,arm,seed,"OfficialVal_original")
        for ds,ytrue,wid in (("OOF",handwritten_y,held),
                             ("OfficialVal",off["y_val"],held)):
            z,p2,c,base,ce,fe=eval_scales(mlp,scale_map,ytrue,ds,wid,arm,seed)
            scale_rows+=z;pairs+=p2;scale_class_rows+=c
            class_errors+=ce;family_errors+=fe
            if ds=="OOF":assert np.array_equal(base,outcome)
            else:assert np.array_equal(base,pp)
        print("FOLD_COMPLETED",arm,seed,held,
              "OOF",round(np.mean(outcome==handwritten_y),6),
              "VAL",round(np.mean(pp==off["y_val"]),6),
              "fit_s",round(sec,2),flush=True)
        del X,y,mlp
    hand_y=np.concatenate([writers[w]["y"] for w in WRITERS])
    pooled_p=np.concatenate(pooled)
    assert len(hand_y)==2560
    output_plots(hand_y,pooled_p,out,arm,seed)
    savecsv(out/"fold_metrics.csv",foldrows)
    savecsv(out/"oof_per_class.csv",class_rows)
    savecsv(out/"per_class_and_scale.csv",scale_class_rows)
    savecsv(out/"pair_confusions.csv",pairs)
    savecsv(out/"per_class_error_rates.csv",class_errors)
    savecsv(out/"family_confusion_rates.csv",family_errors)
    savecsv(out/"scale_sensitivity.csv",scale_rows)
    savecsv(out/"oof_predictions.csv",pred_rows)
    savecsv(out/"training_loss.csv",loss)
    savecsv(out/"training_time.csv",fit)
    overall=metrics(hand_y,pooled_p)
    meta={"experiment":"EXP-CMB-008","stage":"S4_learning_rate","learning_rate_init":learning_rate,"alpha":0.0001,"hidden_layer_sizes":[256],"status":"S4 model trained, selection pending",
          "arm":arm,"seed":seed,"handmade_cv":"Leave-One-Writer-Out 4-Fold",
          "official_n":76800,"official_val_n":19200,"training_handmade_each_fold":1920,
          "oof_n":2560,"scale_interventions":SCALES,"class_balanced_targets":TARGETS,
          "official_train_only_augmentation":True,
          "handmade_train_preprocess":"CoM-only",
          "heldout_and_official_val_preprocess":"CoM-only",
          "augmentation_seed":20261116,"augmentation":info,
          "preregistration":"EXP-CMB-008 S4; width 256, alpha=1e-4; LR 0.0005/0.001/0.002; no Test",
          "hyperparameter_search":True,"only_changed_parameter":"learning_rate_init",
          "fixed_params":{"activation":"relu","solver":"adam","hidden_layer_sizes":[256],"alpha":0.0001,
             "learning_rate_init":learning_rate,"max_iter":200,"early_stopping":False},
          "official_sha256":off["sha256"],
          "handmade_sha256":{str(w):writers[w]["sha256"] for w in WRITERS},
          "official_train_size_binary_probe":official_probe,
          "oof_metrics":overall,"fit_total_seconds":sum(r["fit_seconds"] for r in fit),
          "preprocessing_seconds":preprocess_time,
          "wall_seconds":time.perf_counter()-t0,
          "original_test_used":False,"professor_test_used":False,
          "new_handmade_480_used":False,
          "versions":{"numpy":np.__version__,"scikit_learn":sklearn.__version__},
          "model_sha256":{str(w):hashlib.sha256((out/"models"/f"heldout_{w}.joblib").read_bytes()).hexdigest()
                          for w in WRITERS}}
    (out/"manifest.json").write_text(json.dumps(meta,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    print("COMPLETED",arm,seed,json.dumps(overall),flush=True)

if __name__=="__main__":
    ap=argparse.ArgumentParser()
    ap.add_argument("--arm",choices=("scale30_stroke28",),default="scale30_stroke28")
    ap.add_argument("--learning-rate",type=float,choices=(.0005,.002),required=True)
    ap.add_argument("--seed",type=int,choices=SEEDS,required=True)
    ap.add_argument("--out",type=Path,required=True)
    ar=ap.parse_args()
    run(ar.arm,ar.seed,ar.out,ar.learning_rate)


## exp_cmb008_s5_early_stopping

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_early_stopping.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_early_stopping.py)  
Git blob: `92bef6cdfed15d74a2bb2441814ce00108bbdc3b`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-008 / S5. Freeze (256,), alpha=1e-4; compare early_stopping on S4-selected LR.

S5 retrains ES=True only with the S4-selected LR. ES=False baseline is
from S4 archive if LR is 0.0005 or 0.002; else S1 archive if LR is 0.001.
3 seeds (42/43/44) and 4 writer-held-out folds per candidate.
No Original/Professor Test or new 480. No held-out/OfficialVal training augmentation.

CLI 인자 (정적 추출): `--arm`, `--learning-rate`, `--seed`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb008_s5_early_stopping.py
#!/usr/bin/env python3
"""EXP-CMB-008 / S5. Freeze (256,), alpha=1e-4; compare early_stopping on S4-selected LR.

S5 retrains ES=True only with the S4-selected LR. ES=False baseline is
from S4 archive if LR is 0.0005 or 0.002; else S1 archive if LR is 0.001.
3 seeds (42/43/44) and 4 writer-held-out folds per candidate.
No Original/Professor Test or new 480. No held-out/OfficialVal training augmentation.
"""
from __future__ import annotations
import argparse,csv,hashlib,json,sys,time,warnings
from pathlib import Path
import joblib
import numpy as np
import sklearn
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score,balanced_accuracy_score,roc_auc_score,
                             precision_recall_fscore_support,confusion_matrix)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT=Path(__file__).resolve().parents[4]
CMB=ROOT/"05_model_optimize"/"MLP"/"Combined"
NORMAL=ROOT/"05_model_optimize"/"MLP"/"Normal"
sys.path.insert(0,str(CMB/"scripts"))
sys.path.insert(0,str(NORMAL/"scripts"))
from combined_mlp_writer_cv import (load_official,load_writers,PixelScaler255,
                                    WRITERS,LABELS,V3_SHA,HANDMADE_SHA)
from exp006_size_com_comparison import process,image_geometry
from exp015_scale_stroke_grid import nested_subset
from exp016_high_fraction_grid import scale_base,stroke_jitter
from exp014_balanced_augmentation import balanced_indices
from exp011_012_causal_probes import fixed_scale

SCALES=(.8,.9,1.,1.1,1.2)
PAIRS=(("0","D"),("D","0"),("8","B"),("B","8"),("5","F"),("F","5"),
       ("7","9"),("7","A"),("4","F"))
ARMS={"com_only":(0.,0.),
      "scale10_stroke00":(.1,0.),
      "scale10_stroke14":(.1,.14),
      "scale10_stroke28":(.1,.28),
      "scale20_stroke00":(.2,0.),
      "scale20_stroke14":(.2,.14),
      "scale20_stroke28":(.2,.28),
      "scale30_stroke00":(.3,0.),
      "scale30_stroke14":(.3,.14),
      "scale30_stroke28":(.3,.28)}
TARGETS=(16,18,20,22,24)
SEEDS=(42,43,44)
CLASSES=np.arange(16,dtype=int)

def savecsv(p,records):
    records=list(records)
    if not records:raise RuntimeError("missing rows "+str(p))
    p.parent.mkdir(parents=True,exist_ok=True)
    with p.open("w",newline="",encoding="utf-8-sig") as f:
        wr=csv.DictWriter(f,fieldnames=list(records[0]))
        wr.writeheader();wr.writerows(records)

def metrics(y,p,subset=None):
    labels=CLASSES if subset is None else np.asarray(subset)
    prec,rec,f1,sup=precision_recall_fscore_support(y,p,labels=labels,zero_division=0)
    return {"n":int(len(y)),"correct":int(np.sum(p==y)),
            "accuracy":float(np.mean(p==y)),
            "macro_f1":float(f1.mean()),
            "macro_precision":float(prec.mean()),
            "macro_recall":float(rec.mean())}

def confusion_pairs(y,p,writer,arm,seed,source):
    """All 240 off-diagonal directions, including zero-error pairs.

    Denominator n_actual corresponds to the *actual* (row) class.
    Both counts and conditional percentages are saved; total overall
    accuracy does not replace per-class and per-pair error analysis.
    """
    cm=confusion_matrix(y,p,labels=CLASSES)
    result=[]
    nall=int(len(y))
    nwrong=int(nall-np.trace(cm))
    for actual in CLASSES:
        nactual=int(cm[actual].sum())
        wrong=int(nactual-cm[actual,actual])
        for target in CLASSES:
            if actual==target:continue
            ct=int(cm[actual,target])
            result.append({"arm":arm,"seed":seed,"dataset":source,"writer":writer,
                 "actual":LABELS[actual],"predicted":LABELS[target],
                 "n_actual":nactual,"n_actual_wrong":wrong,
                 "n_dataset":nall,"n_dataset_wrong":nwrong,
                 "errors":ct,"error_rate_within_actual_class":ct/nactual if nactual else 0,
                 "share_of_actual_class_errors":ct/wrong if wrong else 0,
                 "share_of_all_dataset_errors":ct/nwrong if nwrong else 0})
    assert len(result)==16*15
    assert sum(int(r["errors"]) for r in result)==nwrong
    return result

def total_errors_by_class(y,p,writer,arm,seed,source):
    """All 16 true classes: total incorrect / n actual class, plus group switch."""
    cm=confusion_matrix(y,p,labels=CLASSES)
    out=[]
    for a in CLASSES:
        n=int(cm[a].sum());correct=int(cm[a,a]);wrong=n-correct
        family_swap=int(cm[a,10:].sum()) if a<10 else int(cm[a,:10].sum())
        out.append({"arm":arm,"seed":seed,"dataset":source,"writer":writer,
           "actual":LABELS[a],"n":n,"correct":correct,"wrong":wrong,
           "class_error_rate":wrong/n if n else 0,
           "opposite_family_errors":family_swap,
           "opposite_family_error_rate":family_swap/n if n else 0})
    assert len(out)==16
    return out

def family_error_breakdown(y,p,writer,arm,seed,source):
    """Digits→letters and letters→digits alongside within-family mistakes."""
    cm=confusion_matrix(y,p,labels=CLASSES)
    d2l=int(cm[:10,10:].sum());l2d=int(cm[10:,:10].sum())
    d2d=int(cm[:10,:10].sum()-np.trace(cm[:10,:10]))
    l2l=int(cm[10:,10:].sum()-np.trace(cm[10:,10:]))
    return [{"arm":arm,"seed":seed,"dataset":source,"writer":writer,
      "direction":key,"errors":val,"n_actual_family":n,
      "error_rate":val/n if n else 0}
      for key,val,n in (("digit_to_letter",d2l,int(cm[:10].sum())),
                        ("letter_to_digit",l2d,int(cm[10:].sum())),
                        ("digit_to_digit",d2d,int(cm[:10].sum())),
                        ("letter_to_letter",l2l,int(cm[10:].sum())))]

def geom_rows(name,x,y,arm,seed):
    geo=image_geometry(x)
    out=[]
    for nameg,mask in (("digits",y<10),("letters",y>=10)):
        z={"arm":arm,"seed":seed,"dataset":name,"group":nameg,
           "n":int(mask.sum())}
        for field in ("long_side","width","height","mass"):
            zz=np.asarray(geo[field])[mask]
            z[field+"_mean"]=float(zz.mean())
            z[field+"_std"]=float(zz.std())
        out.append(z)
    return out

def binary_size_probe(xtrain,ytrain,xval,yval):
    train=image_geometry(xtrain)["long_side"].reshape(-1,1)
    val=image_geometry(xval)["long_side"].reshape(-1,1)
    model=Pipeline([("std",StandardScaler()),
      ("logistic",LogisticRegression(class_weight="balanced",random_state=42,max_iter=1000))])
    model.fit(train,(ytrain>=10).astype(int))
    q=model.predict_proba(val)[:,1]
    return {"bal_acc":float(balanced_accuracy_score(yval>=10,q>=.5)),
            "auc":float(roc_auc_score(yval>=10,q))}

def prepareraw(official,writers,arm):
    ratio,stroke=ARMS[arm]
    raw=official["X_train"]
    yy=official["y_train"]
    # CoM-only is exactly the EXP-CMB-004 reference condition.
    main=process(raw,"com",20)
    changed={"method":arm,"requested_ratio":ratio,"actual_ratio":0.,
             "n_aug":0,"n_stroke_changed":0,"n_stroke_rejected":0,
             "official_train_rows":len(raw)}
    targets=[]
    if ratio:
        ids,target_map=nested_subset(yy,ratio)  # Normal EXP015 nested low-ratio group
        half,_=balanced_indices(yy,seed=20261116)
        augmented=scale_base(raw,ids,target_map)
        augmented,stroke_stats=stroke_jitter(augmented,stroke,ids,half)
        main[ids]=process(augmented,"com",20)
        changed.update({"n_aug":len(ids),"actual_ratio":len(ids)/len(yy),
             "n_stroke_changed":stroke_stats["modified"],
             "n_stroke_rejected":stroke_stats["rejected"]})
        for cl in CLASSES:
            for length in TARGETS:
                ct=int(np.sum((yy[ids]==cl)&(target_map[ids]==length)))
                targets.append({"arm":arm,"class":LABELS[cl],"target_size":length,
                                "train_images_assigned":ct})
                assert ct>=1
        for cl in CLASSES:
            cc=[x["train_images_assigned"] for x in targets if x["class"]==LABELS[cl]]
            assert max(cc)-min(cc)<=1
        assert abs(len(ids)/len(yy)-ratio)<.002
    else:
        for cl in CLASSES:
            for length in TARGETS:
                targets.append({"arm":arm,"class":LABELS[cl],"target_size":length,
                                "train_images_assigned":0})
    transformed_w={w:process(writers[w]["X"],"com",20) for w in WRITERS}
    assert main.shape==(76800,784) and len(transformed_w)==4
    return main,transformed_w,changed,targets

def make_scaled(official,writers):
    ref={}
    for scale in SCALES:
        # exact Normal EXP012 intervention; raw image BBox-centered affine, then CoM
        ref[("official",scale)]=process(fixed_scale(official["X_val"],scale),"com",20)
        for w in WRITERS:
            ref[(w,scale)]=process(fixed_scale(writers[w]["X"],scale),"com",20)
    return ref

def eval_scales(model,raw_map,y,dataset,writer,arm,seed):
    rows=[]; pair_rows=[]; focus=[]; class_errors=[]; family_errors=[]
    probs={}
    for sc in SCALES:
        xx=raw_map[(writer if dataset=="OOF" else "official",sc)]
        q=np.asarray(model.predict_proba(xx),dtype=np.float64)
        assert np.array_equal(model.named_steps["model"].classes_,CLASSES)
        p=q.argmax(axis=1).astype(int)
        letter=q[:,10:].sum(axis=1)
        probs[sc]=(p,letter)
    _,base_letter=probs[1.]
    base_family=base_letter>=.5
    for sc in SCALES:
        p,letter=probs[sc]
        for group,z,cls in (("all",np.ones(len(y),dtype=bool),CLASSES),
             ("digits",y<10,np.arange(10)),("letters",y>=10,np.arange(10,16))):
            m=metrics(y[z],p[z],cls)
            flip=(letter[z]>=.5)!=(base_family[z])
            rows.append({"arm":arm,"seed":seed,"writer":writer,"dataset":dataset,
               "scale":sc,"group":group,**m,
               "family_flip":float(flip.mean()),
               "mean_abs_delta_p_letter":float(np.abs(letter[z]-base_letter[z]).mean()),
               "mean_p_letter":float(letter[z].mean())})
        for cls_i in CLASSES:
            z=y==cls_i
            pz=p[z];ll=letter[z];bb=base_letter[z]
            focus.append({"arm":arm,"seed":seed,"writer":writer,"dataset":dataset,
                "scale":sc,"actual":LABELS[cls_i],"n":int(z.sum()),
                "correct":int(np.sum(pz==cls_i)),
                "accuracy":float(np.mean(pz==cls_i)),
                "family_flip":float(np.mean((ll>=.5)!=(bb>=.5))),
                "mean_abs_delta_p_letter":float(np.mean(np.abs(ll-bb))),
                "mean_p_letter":float(ll.mean())})
        sc_name=dataset+"_scale_"+str(sc)
        pair_rows+=confusion_pairs(y,p,writer,arm,seed,sc_name)
        class_errors+=total_errors_by_class(y,p,writer,arm,seed,sc_name)
        family_errors+=family_error_breakdown(y,p,writer,arm,seed,sc_name)
    return rows,pair_rows,focus,probs[1.][0],class_errors,family_errors

def output_plots(y,p,out,arm,seed):
    cm=confusion_matrix(y,p,labels=CLASSES)
    for k in ("counts","row_normalized"):
        v=cm if k=="counts" else 100*cm/np.maximum(cm.sum(axis=1,keepdims=True),1)
        fig,ax=plt.subplots(figsize=(10,8))
        im=ax.imshow(v,cmap="viridis",vmin=0,vmax=160 if k=="counts" else 100)
        fig.colorbar(im,ax=ax,fraction=.045,pad=.04)
        ax.set(xticks=np.arange(16),yticks=np.arange(16),
               xticklabels=list(LABELS),yticklabels=list(LABELS),
               xlabel="Predicted",ylabel="True",
               title=f"EXP-CMB-005 {arm} seed={seed} Combined OOF {k}")
        fig.tight_layout()
        fig.savefig(out/f"OOF_confusion_{k}.png",dpi=145)
        plt.close(fig)

def run(arm,seed,out,learning_rate):
    if arm!="scale30_stroke28" or seed not in SEEDS or learning_rate not in (.0005,.001,.002):raise ValueError((arm,seed,learning_rate))
    if out.exists(): raise FileExistsError(out)
    out.mkdir(parents=True)
    (out/"models").mkdir()
    t0=time.perf_counter()
    original=ROOT/"02_data"/"datasets"/"MNIST_EMNIST_official_split_112000.npz"
    handmade=ROOT/"02_data"/"Hand_made"
    off=load_official(original)
    writers=load_writers(handmade)
    assert off["sha256"]==V3_SHA
    for w in WRITERS:assert writers[w]["sha256"]==HANDMADE_SHA[w]
    assert len(off["X_train"])==76800 and len(off["X_val"])==19200
    tpre=time.perf_counter()
    xtrain_off,xwriters,info,targets=prepareraw(off,writers,arm)
    xval=process(off["X_val"],"com",20)
    scale_map=make_scaled(off,writers)
    preprocess_time=time.perf_counter()-tpre
    official_probe=binary_size_probe(xtrain_off,off["y_train"],xval,off["y_val"])
    savecsv(out/"target_assignments.csv",targets)
    features=geom_rows("official_train",xtrain_off,off["y_train"],arm,seed)
    features+=geom_rows("official_val",xval,off["y_val"],arm,seed)
    for w in WRITERS:features+=geom_rows("writer_"+str(w),xwriters[w],writers[w]["y"],arm,seed)
    savecsv(out/"geometry_distributions.csv",features)
    foldrows=[];pairs=[];class_errors=[];family_errors=[];scale_rows=[];class_rows=[];scale_class_rows=[];pred_rows=[];loss=[];fit=[]
    pooled=[]
    for held in WRITERS:
        trwriters=[w for w in WRITERS if w!=held]
        X=np.concatenate([xtrain_off]+[xwriters[w] for w in trwriters])
        y=np.concatenate([off["y_train"]]+[writers[w]["y"] for w in trwriters])
        assert X.shape==(78720,784) and len(y)==78720
        mlp=Pipeline([("scale_0_1",PixelScaler255()),
                      ("model",MLPClassifier(random_state=seed,hidden_layer_sizes=(256,),alpha=0.0001,learning_rate_init=learning_rate,early_stopping=True,validation_fraction=0.10,n_iter_no_change=10))])
        started=time.perf_counter()
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            mlp.fit(X,y)
        sec=time.perf_counter()-started
        mlpnet=mlp.named_steps["model"]
        assert mlpnet.n_features_in_==784
        for it,lossval in enumerate(mlpnet.loss_curve_):
            loss.append({"arm":arm,"seed":seed,"heldout":held,
                        "iteration":it+1,"loss":float(lossval)})
        fit.append({"arm":arm,"seed":seed,"heldout":held,
                    "train_n":len(y),"heldout_n":640,
                    "fit_seconds":sec,"iterations":mlpnet.n_iter_,
                    "last_loss":float(mlpnet.loss_),
                    "warnings":" | ".join(str(w.message) for w in caught)})
        fout=out/"models"/f"heldout_{held}.joblib"
        joblib.dump(mlp,fout,compress=3)
        handwritten_y=writers[held]["y"]
        outcome=mlp.predict(xwriters[held]).astype(int)
        pp=mlp.predict(xval).astype(int)
        for ds,ytrue,pred in (("OOF",handwritten_y,outcome),
                              ("OfficialVal",off["y_val"],pp)):
            maskset=(("all",np.ones(len(ytrue),bool),CLASSES),
                     ("digits",ytrue<10,np.arange(10)),
                     ("letters",ytrue>=10,np.arange(10,16)))
            for g,z,classes in maskset:
                foldrows.append({"arm":arm,"seed":seed,"heldout":held,"dataset":ds,
                      "group":g,**metrics(ytrue[z],pred[z],classes)})
        prec,rec,f1,_=precision_recall_fscore_support(handwritten_y,outcome,
              labels=CLASSES,zero_division=0)
        for ci in CLASSES:
            z=handwritten_y==ci
            class_rows.append({"arm":arm,"seed":seed,"writer":held,
               "class":LABELS[ci],"n":int(z.sum()),"correct":int(np.sum(outcome[z]==ci)),
               "precision":float(prec[ci]),"recall":float(rec[ci]),"f1":float(f1[ci])})
        for i in range(640):
            pred_rows.append({"arm":arm,"seed":seed,"writer":held,"index":i,
                "actual":LABELS[int(handwritten_y[i])],
                "pred":LABELS[int(outcome[i])]})
        pooled.append(outcome)
        pairs+=confusion_pairs(handwritten_y,outcome,held,arm,seed,"OOF_original")
        pairs+=confusion_pairs(off["y_val"],pp,held,arm,seed,"OfficialVal_original")
        class_errors+=total_errors_by_class(handwritten_y,outcome,held,arm,seed,"OOF_original")
        class_errors+=total_errors_by_class(off["y_val"],pp,held,arm,seed,"OfficialVal_original")
        family_errors+=family_error_breakdown(handwritten_y,outcome,held,arm,seed,"OOF_original")
        family_errors+=family_error_breakdown(off["y_val"],pp,held,arm,seed,"OfficialVal_original")
        for ds,ytrue,wid in (("OOF",handwritten_y,held),
                             ("OfficialVal",off["y_val"],held)):
            z,p2,c,base,ce,fe=eval_scales(mlp,scale_map,ytrue,ds,wid,arm,seed)
            scale_rows+=z;pairs+=p2;scale_class_rows+=c
            class_errors+=ce;family_errors+=fe
            if ds=="OOF":assert np.array_equal(base,outcome)
            else:assert np.array_equal(base,pp)
        print("FOLD_COMPLETED",arm,seed,held,
              "OOF",round(np.mean(outcome==handwritten_y),6),
              "VAL",round(np.mean(pp==off["y_val"]),6),
              "fit_s",round(sec,2),flush=True)
        del X,y,mlp
    hand_y=np.concatenate([writers[w]["y"] for w in WRITERS])
    pooled_p=np.concatenate(pooled)
    assert len(hand_y)==2560
    output_plots(hand_y,pooled_p,out,arm,seed)
    savecsv(out/"fold_metrics.csv",foldrows)
    savecsv(out/"oof_per_class.csv",class_rows)
    savecsv(out/"per_class_and_scale.csv",scale_class_rows)
    savecsv(out/"pair_confusions.csv",pairs)
    savecsv(out/"per_class_error_rates.csv",class_errors)
    savecsv(out/"family_confusion_rates.csv",family_errors)
    savecsv(out/"scale_sensitivity.csv",scale_rows)
    savecsv(out/"oof_predictions.csv",pred_rows)
    savecsv(out/"training_loss.csv",loss)
    savecsv(out/"training_time.csv",fit)
    overall=metrics(hand_y,pooled_p)
    meta={"experiment":"EXP-CMB-008","stage":"S5_early_stopping","learning_rate_init":learning_rate,"alpha":0.0001,"hidden_layer_sizes":[256],"status":"S5 model trained, selection pending",
          "arm":arm,"seed":seed,"handmade_cv":"Leave-One-Writer-Out 4-Fold",
          "official_n":76800,"official_val_n":19200,"training_handmade_each_fold":1920,
          "oof_n":2560,"scale_interventions":SCALES,"class_balanced_targets":TARGETS,
          "official_train_only_augmentation":True,
          "handmade_train_preprocess":"CoM-only",
          "heldout_and_official_val_preprocess":"CoM-only",
          "augmentation_seed":20261116,"augmentation":info,
          "preregistration":"EXP-CMB-008 S5; width 256, alpha=1e-4, S4-selected LR, ES True vs False; no Test",
          "hyperparameter_search":True,"only_changed_parameter":"early_stopping",
          "fixed_params":{"activation":"relu","solver":"adam","hidden_layer_sizes":[256],"alpha":0.0001,
             "learning_rate_init":learning_rate,"max_iter":200,"early_stopping":True},
          "early_stopping_internal_validation_fraction":0.10,"early_stopping_n_iter_no_change":10,
          "official_sha256":off["sha256"],
          "handmade_sha256":{str(w):writers[w]["sha256"] for w in WRITERS},
          "official_train_size_binary_probe":official_probe,
          "oof_metrics":overall,"fit_total_seconds":sum(r["fit_seconds"] for r in fit),
          "preprocessing_seconds":preprocess_time,
          "wall_seconds":time.perf_counter()-t0,
          "original_test_used":False,"professor_test_used":False,
          "new_handmade_480_used":False,
          "versions":{"numpy":np.__version__,"scikit_learn":sklearn.__version__},
          "model_sha256":{str(w):hashlib.sha256((out/"models"/f"heldout_{w}.joblib").read_bytes()).hexdigest()
                          for w in WRITERS}}
    (out/"manifest.json").write_text(json.dumps(meta,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    print("COMPLETED",arm,seed,json.dumps(overall),flush=True)

if __name__=="__main__":
    ap=argparse.ArgumentParser()
    ap.add_argument("--arm",choices=("scale30_stroke28",),default="scale30_stroke28")
    ap.add_argument("--learning-rate",type=float,choices=(.0005,.001,.002),required=True)
    ap.add_argument("--seed",type=int,choices=SEEDS,required=True)
    ap.add_argument("--out",type=Path,required=True)
    ar=ap.parse_args()
    run(ar.arm,ar.seed,ar.out,ar.learning_rate)


## exp_cmb008_s4_s5_aggregate

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb008_s4_s5_aggregate.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s4_s5_aggregate.py)  
Git blob: `1a6a6772c4ada202b531b80c8dc3d16b97243ea2`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-008 S4/S5 sequential, evidence-first audit.

S4: LR=.0005/.001/.002 with frozen Size30%-Stroke28%, CoM, (256,), alpha=.0001,
    Adam/ReLU, early_stopping=False, with S1 original LR=.001 as exact baseline.
S5: keep LR selected by S4; compare early_stopping=False to True (training-only
    internal validation). Reuse S4/S1 ES=False models. Never touch independent Test.

CLI 인자 (정적 추출): `--stage`, `--s1-reference`, `--s4-new`, `--s5-new`, `--s4-decision`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb008_s4_s5_aggregate.py
#!/usr/bin/env python3
"""EXP-CMB-008 S4/S5 sequential, evidence-first audit.

S4: LR=.0005/.001/.002 with frozen Size30%-Stroke28%, CoM, (256,), alpha=.0001,
    Adam/ReLU, early_stopping=False, with S1 original LR=.001 as exact baseline.
S5: keep LR selected by S4; compare early_stopping=False to True (training-only
    internal validation). Reuse S4/S1 ES=False models. Never touch independent Test.
"""
from __future__ import annotations
import argparse,csv,json,statistics
from pathlib import Path
from exp_cmb008_s1_aggregate import (
    SEEDS,WRITERS,readcsv,summarize,check,target_under_5percent,paired,pct
)

PAIRS=("D->0","B->8","8->B","0->D")
LR_KEYS=("0.0005","0.001","0.002")
REQUIRED_FILES=("manifest.json","oof_predictions.csv","fold_metrics.csv",
                "per_class_and_scale.csv","training_time.csv","training_loss.csv")
def read_run(directory,seed,lr,stage,early_stopping):
    assert directory.is_dir(),f"Artifact absent: {directory}"
    for required in REQUIRED_FILES:assert (directory/required).is_file(),directory/required
    m=json.loads((directory/"manifest.json").read_text(encoding="utf-8"))
    assert m["experiment"]=="EXP-CMB-008" and m["arm"]=="scale30_stroke28"
    assert int(m["seed"])==seed and m["augmentation_seed"]==20261116
    assert m["handmade_train_preprocess"]=="CoM-only"
    assert m["heldout_and_official_val_preprocess"]=="CoM-only"
    assert m["official_train_only_augmentation"]
    assert not any(m[x] for x in ("original_test_used","professor_test_used","new_handmade_480_used"))
    assert m["hidden_layer_sizes"]==[256] if stage!="S1_width" else m["width"]==256
    expected={"activation":"relu","solver":"adam","alpha":.0001,
              "learning_rate_init":float(lr),"max_iter":200,"early_stopping":early_stopping}
    if stage=="S1_width":
        assert lr=="0.001" and not early_stopping
        assert m["stage"]=="S1_width" and m["only_changed_parameter"]=="hidden_layer_sizes"
    else:
        expected["hidden_layer_sizes"]=[256]
        assert m["stage"]==stage and m["only_changed_parameter"]==(
            "learning_rate_init" if stage=="S4_learning_rate" else "early_stopping")
        assert abs(float(m["learning_rate_init"])-float(lr))<1e-12
        if stage=="S5_early_stopping":
            assert early_stopping
            assert m["early_stopping_internal_validation_fraction"]==.1
            assert m["early_stopping_n_iter_no_change"]==10
    assert m["fixed_params"]==expected,(directory,m["fixed_params"],expected)
    oof=readcsv(directory/"oof_predictions.csv")
    assert len(oof)==2560 and len({(int(x["writer"]),int(x["index"])) for x in oof})==2560
    assert {int(x["writer"]) for x in oof}==set(WRITERS)
    assert all(x["actual"] in "0123456789ABCDEF" and x["pred"] in "0123456789ABCDEF" for x in oof)
    vals=[x for x in readcsv(directory/"fold_metrics.csv") if x["dataset"]=="OfficialVal" and x["group"]=="all"]
    assert len(vals)==4 and all(int(x["n"])==19200 for x in vals)
    scales=[x for x in readcsv(directory/"per_class_and_scale.csv") if
            x["dataset"] in ("OOF","OfficialVal") and float(x["scale"]) in (.8,1.2)]
    assert scales
    times=readcsv(directory/"training_time.csv")
    assert len(times)==4 and all(int(x["train_n"])==78720 for x in times)
    loss=readcsv(directory/"training_loss.csv")
    assert loss
    byfold={}
    for r in loss:
        k=int(r["heldout"])
        byfold.setdefault(k,[]).append((int(r["iteration"]),float(r["loss"])))
    assert set(byfold)==set(WRITERS)
    assert all(byfold[w][0][0]==1 and len(byfold[w])==int(next(x for x in times if int(x["heldout"])==w)["iterations"]) for w in WRITERS)
    return {"m":m,"oof":oof,"val":vals,"scales":scales,"times":times,
            "loss_start":statistics.mean(byfold[w][0][1] for w in WRITERS),
            "loss_end":statistics.mean(byfold[w][-1][1] for w in WRITERS)}
def load_group(paths,lr,stage,es):
    return {seed:read_run(paths[seed],seed,lr,stage,es) for seed in SEEDS}
def verify_sha(groups):
    baseline=next(iter(groups.values()))
    for key,data in groups.items():
        for seed in SEEDS:
            assert data[seed]["m"]["official_sha256"]==baseline[seed]["m"]["official_sha256"],(key,seed,"official SHA mismatch")
            assert data[seed]["m"]["handmade_sha256"]==baseline[seed]["m"]["handmade_sha256"],(key,seed,"handmade SHA mismatch")
            assert data[seed]["m"]["augmentation"]["requested_ratio"]==.3
def rank(base,metrics):
    guards={key:check(base,val) for key,val in metrics.items()}
    strict=[k for k in metrics if target_under_5percent(metrics[k]) and all(guards[k].values())]
    safe=[k for k in metrics if all(guards[k].values())]
    assert safe, "The baseline itself must pass non-regression gates"
    pool=strict or safe
    maximum=max(metrics[k]["oof"] for k in pool)
    near=[k for k in pool if metrics[k]["oof"]>=maximum-.002-1e-12]
    near.sort(key=lambda k:(metrics[k]["pairs"]["D->0"],
        sum(metrics[k]["pairs"][x] for x in PAIRS if x!="D->0"),
        -metrics[k]["oof"],metrics[k]["seed_std"],-metrics[k]["val"],
        float(k) if k in LR_KEYS else (0 if k=="False" else 1)))
    selected=near[0]
    return guards,strict,safe,selected
def build_rows(metrics,guards):
    results=[]
    for key,val in metrics.items():
        gate=guards[key]
        record={"candidate":key,"oof_accuracy":val["oof"],
          "oof_misclassification":1-val["oof"],"official_val_accuracy":val["val"],
          "seed_std":val["seed_std"],"worst_writer_accuracy":min(val["writers"].values()),
          "D_to_0":val["pairs"]["D->0"],"B_to_8":val["pairs"]["B->8"],
          "8_to_B":val["pairs"]["8->B"],"0_to_D":val["pairs"]["0->D"],
          "D_to_0_rate":val["pairs"]["D->0"]/480,"B_to_8_rate":val["pairs"]["B->8"]/480,
          "8_to_B_rate":val["pairs"]["8->B"]/480,"0_to_D_rate":val["pairs"]["0->D"]/480,
          "class_0_error_rate":val["classerror"]["0"],"class_8_error_rate":val["classerror"]["8"],
          "class_B_error_rate":val["classerror"]["B"],"class_D_error_rate":val["classerror"]["D"],
          "oof_max_family_flip":val["flip"]["OOF"],"val_max_family_flip":val["flip"]["OfficialVal"],
          "mean_iterations":val["mean_n_iter"],"mean_fit_seconds":val["mean_fit_seconds"],
          "warning_folds":val["warnings"],"all_four_directions_under_5pct":target_under_5percent(val),
          "all_fallback_safeguards_pass":all(gate.values()),
          "failed_gates":";".join(k for k,v in gate.items() if not v)}
        results.append(record)
    return results
def save_csv(p,rows):
    with p.open("w",newline="",encoding="utf-8-sig") as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0]))
        writer.writeheader();writer.writerows(rows)
def format_details(stage,baseline_key,selected_key,groups,metrics,guards,strict,safe,paired_errors,loss_by_key):
    return {"experiment":"EXP-CMB-008","stage":stage,"selected":selected_key,
      "status":"TARGET_MET" if strict else ("BASELINE_RETAINED" if selected_key==baseline_key else "FALLBACK_IMPROVED"),
      "selected_is_independent_final_test":False,
      "all_four_directions_under_5pct_candidates":strict,
      "safe_fallback_candidates":safe,
      "target_rate_each_direction":"strictly below 5%", "max_count_per_direction":23,
      "actual_class_denominator":480,
      "criterion":"if none <5% with gates, maximize OOF accuracy with no regression in D->0/B->8/8->B/0->D, errors of 0/8/B/D, or overfit safeguards",
      "scores":metrics,"gates":guards,"paired_corrected_and_induced":paired_errors,
      "average_first_and_last_training_loss":loss_by_key,
      "same_official_and_handmade_sha_confirmed":True,
      "no_independent_test_used":True,"training_seeds":list(SEEDS),
      "heldout_writers":list(WRITERS)}
def write_report(stage,baseline_key,selected,rows,metrics,guards,strict,paired_errors,loss,extra):
    name="S4 Learning Rate" if stage=="S4_learning_rate" else "S5 Early Stopping"
    why=(
      "S3에서 L2 강도를 높이자 일부 정확도는 증가했으나 D→0 등 핵심 오분류가 악화되었다. alpha=1e-4, (256,)을 유지한 채 Adam 초기 학습률을 변화시켜 수렴·정확도·오분류를 분리 비교했다."
      if stage=="S4_learning_rate" else
      "S4에서 실제 선정된 LR을 고정하고 Training Fold 내부 10%만 사용하는 Early Stopping을 적용했다. 같은 LR의 ES=False 원본 모델을 재사용해 조기 종료 효과 및 과소·과적합을 비교했다.")
    md=[f"# EXP-CMB-008 — {name} 실측·결정",
        "",f"> **단계: {stage} · 판단: {'4방향 5% 목표 달성' if strict else '4방향 5% 목표 미달'}**",
        f"> 기존 기준 {baseline_key} / 선택 {selected}; 이는 개발용 OOF 기반 선택이며 최종 독립 Test의 성능이 아니다.","",
        "## 이전 실험 → 이번 실험: 후보를 정한 이유",why,
        "고정: Official Train의 Size30%/Stroke28% 증강, CoM, 단층 (256,), alpha=1e-4, ReLU/Adam, 3 Seed × 4 Writer CV, 데이터 SHA. "
        +( "ES=False 고정; LR만 0.0005/0.001/0.002 비교." if stage=="S4_learning_rate" else
           "S4에서 선정한 LR을 고정하고 ES=False/True만 변경."),"",
        "## 전체 정확도와 중요 혼동 방향 (실제 클래스당 480회 예측)",
        "| 후보 | OOF 정확도 | OOF 오분류율 | OfficialVal | D→0 | B→8 | 8→B | 0→D | 네 방향 <5% | 비악화·일반화 Gate |",
        "|---|---:|---:|---:|---:|---:|---:|---:|---|---|"]
    for r in rows:
        def count(k):return str(r[k])+" ("+pct(r[k+"_rate"])+")"
        md.append("| "+str(r["candidate"])+" | "+pct(r["oof_accuracy"])+" | "+pct(r["oof_misclassification"])+" | "+
           pct(r["official_val_accuracy"])+" | "+" | ".join(count(k) for k in ("D_to_0","B_to_8","8_to_B","0_to_D"))+
           " | "+("통과" if r["all_four_directions_under_5pct"] else "미달")+
           " | "+(r["failed_gates"] or "통과")+" |")
    md+=["","## 오버피팅·클래스 오류 및 수렴 증거",
         "| 후보 | 실제0 오류 | 실제8 오류 | 실제B 오류 | 실제D 오류 | 최저 Writer OOF | Seed 표준편차 | 최대 OOF Flip | 최대 OfficialVal Flip | 반복횟수 | 경고 Fold |",
         "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|"]
    for r in rows:
        md.append("| "+str(r["candidate"])+" | "+" | ".join(pct(r[f"class_{cl}_error_rate"]) for cl in ("0","8","B","D"))+
          " | "+pct(r["worst_writer_accuracy"])+" | "+pct(r["seed_std"])+
          " | "+pct(r["oof_max_family_flip"])+" | "+pct(r["val_max_family_flip"])+
          " | "+f'{r["mean_iterations"]:.1f}'+" | "+str(r["warning_folds"])+" |")
    md+=["","### 4명의 보류 Writer별 OOF 정확도",
        "| 후보 | W41 | W42 | W43 | W44 | Training loss 초기 평균 | Training loss 마지막 평균 | 평균 Fold 시간(초) |",
        "|---|---:|---:|---:|---:|---:|---:|---:|"]
    for key in metrics:
        m=metrics[key]
        md.append("| "+str(key)+" | "+" | ".join(pct(m["writers"][str(w)]) for w in WRITERS)+
          " | "+f'{loss[key]["first"]:.6f}'+" | "+f'{loss[key]["last"]:.6f}'+
          " | "+f'{m["mean_fit_seconds"]:.1f}'+" |")
    md+=["","### 같은 이미지: 기존 오답 교정과 새 오답",
         "| 후보 | 기존 오답 교정 | 기존 정답 신규 오답 | 순개선 |",
         "|---|---:|---:|---:|"]
    for key,pair in paired_errors.items():
        corr,induced=pair
        md.append("| "+key+" | "+str(corr)+" | "+str(induced)+" | "+str(corr-induced)+" |")
    md+=["","## 채택·탈락의 명시적 근거",
        f"- **이번 단계 채택값: {selected}**.",
        f"- **4방향 각각 5% 미만 및 Gate 통과:** {', '.join(strict) if strict else '없음'}. 목표는 아직 달성하지 못했다면 이를 우회해 성공으로 표기하지 않는다.",
        "- **단계별 후보 판정:**"]
    for r in rows:
        k=r["candidate"];why="비교 기준선" if k==baseline_key else (r["failed_gates"] or "안전 Gate 모두 통과")
        md.append(f"  - **{k}:** {why}. D→0 {r['D_to_0']}건·OOF {pct(r['oof_accuracy'])}·OfficialVal {pct(r['official_val_accuracy'])}.")
    md+=["",("- **다음 S5:** 방금 선택한 LR을 고정하고 Training-only Early Stopping=False/True를 비교. 이 단계의 결과를 보지 않고 ES를 먼저 결정하지 않는다."
        if stage=="S4_learning_rate" else
        "- **S1~S5 이후:** Size30/Stroke28/CoM·(256,)·alpha=1e-4·이번 LR 및 ES 선택값을 잠정 동결. 전체 실험의 개발용 결과로 정리하며 최종 Test는 여전히 미사용."),
        "- **과적합 주의:** 같은 4 Writer를 반복 비교해 선택 편향 가능성이 남는다. Internal Early Stopping과 ConvergenceWarning 무발생만으로 과적합 부재를 증명하지 않는다.",
        "- **데이터 경계:** Original Test·교수님 Test·신규 Hand-made 480장 모두 HP 선택 과정에서 미사용.",
        ""]
    return "\n".join(md)
def main():
    ap=argparse.ArgumentParser()
    ap.add_argument("--stage",choices=("s4","s5"),required=True)
    ap.add_argument("--s1-reference",type=Path,required=True)
    ap.add_argument("--s4-new",type=Path)
    ap.add_argument("--s5-new",type=Path)
    ap.add_argument("--s4-decision",type=Path)
    ap.add_argument("--out",type=Path,required=True)
    a=ap.parse_args()
    s1={s:a.s1_reference/f"cmb008-w256-s{s}" for s in SEEDS}
    if a.stage=="s4":
        assert a.s4_new is not None
        groups={}
        for lr in LR_KEYS:
            paths=s1 if lr=="0.001" else {s:a.s4_new/f"cmb008-s4-lr{lr}-s{s}" for s in SEEDS}
            groups[lr]=load_group(paths,lr,"S1_width" if lr=="0.001" else "S4_learning_rate",False)
        verify_sha(groups)
        metrics={key:summarize(key,run) for key,run in groups.items()}
        baseline_key="0.001"
        assert abs(metrics[baseline_key]["oof"]-.9221354166666667)<1e-9
        assert abs(metrics[baseline_key]["val"]-.9704427083333333)<1e-9
        gates,strict,safe,selected=rank(metrics[baseline_key],metrics)
        paired_errors={key:paired(groups[baseline_key],g) for key,g in groups.items() if key!=baseline_key}
        loss={key:{"first":statistics.mean(groups[key][s]["loss_start"] for s in SEEDS),
                   "last":statistics.mean(groups[key][s]["loss_end"] for s in SEEDS)} for key in groups}
        stage="S4_learning_rate"
        tag="S4"
        decision_extra={"selected_learning_rate":selected,"alpha_fixed":.0001,
            "structure_fixed":[256],"early_stopping_fixed":False,
            "new_fits":24,"baseline_reused_models":12}
    else:
        assert a.s4_new is not None and a.s5_new is not None and a.s4_decision is not None
        s4=json.loads(a.s4_decision.read_text(encoding="utf-8"))
        assert s4["stage"]=="S4_learning_rate"
        lr=s4["selected_learning_rate"]
        assert lr in LR_KEYS
        paths=s1 if lr=="0.001" else {s:a.s4_new/f"cmb008-s4-lr{lr}-s{s}" for s in SEEDS}
        groups={"False":load_group(paths,lr,"S1_width" if lr=="0.001" else "S4_learning_rate",False),
                "True":load_group({s:a.s5_new/f"cmb008-s5-lr{lr}-s{s}" for s in SEEDS},lr,"S5_early_stopping",True)}
        verify_sha(groups)
        metrics={key:summarize(key,run) for key,run in groups.items()}
        baseline_key="False"
        expected=s4["scores"][lr]
        assert abs(metrics[baseline_key]["oof"]-expected["oof"])<1e-12, "S5 baseline must match S4 selected archive"
        assert abs(metrics[baseline_key]["val"]-expected["val"])<1e-12
        gates,strict,safe,selected=rank(metrics[baseline_key],metrics)
        paired_errors={"True":paired(groups["False"],groups["True"])}
        loss={key:{"first":statistics.mean(groups[key][s]["loss_start"] for s in SEEDS),
                   "last":statistics.mean(groups[key][s]["loss_end"] for s in SEEDS)} for key in groups}
        stage="S5_early_stopping"
        tag="S5"
        decision_extra={"selected_learning_rate":lr,
                        "selected_early_stopping":selected=="True",
                        "s4_selected_learning_rate_from_verified_decision":True,
                        "internal_training_validation_fraction_if_enabled":.1,
                        "alpha_fixed":.0001,"structure_fixed":[256],
                        "new_fits":12,"baseline_reused_models":12}
    report=format_details(stage,baseline_key,selected,groups,metrics,gates,strict,safe,paired_errors,loss)
    report.update(decision_extra)
    a.out.mkdir(parents=True,exist_ok=True)
    (a.out/f"{tag}_details.json").write_text(json.dumps(report,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    rows=build_rows(metrics,gates)
    save_csv(a.out/f"{tag}_comparison.csv",rows)
    text=write_report(stage,baseline_key,selected,rows,metrics,gates,strict,paired_errors,loss,decision_extra)
    (a.out/f"{tag}_Stage_Results.md").write_text(text,encoding="utf-8")
    print(f"{tag}_DECISION",json.dumps({"status":report["status"],"selected":selected,
            "strict":strict,"safe":safe,"rows":rows},ensure_ascii=False),flush=True)
if __name__=="__main__":main()


## exp_cmb008_s4_s5_finalize

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb008_s4_s5_finalize.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s4_s5_finalize.py)  
Git blob: `c8a2040fc323c040ff3963607edf9d74accf234e`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

Finalize CMB-008 S1-S5 only from verified stage results; no model deployment.



In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb008_s4_s5_finalize.py
#!/usr/bin/env python3
"""Finalize CMB-008 S1-S5 only from verified stage results; no model deployment."""
import argparse,json
from pathlib import Path

def pct(x):return f"{x*100:.3f}%"
def with_banner(s,body):
    a="<!-- CMB008_S4S5_FINAL_START -->"
    b="<!-- CMB008_S4S5_FINAL_END -->"
    if a in s and b in s:return s[:s.index(a)]+a+"\n"+body+"\n"+s[s.index(b):]
    return s.rstrip()+"\n\n"+a+"\n"+body+"\n"+b+"\n"
def table(stage):
    rows=["| 조건 | OOF 정확도 | OfficialVal | D→0 | B→8 | 8→B | 0→D | 4방향 <5% | 악화 Gate |",
          "|---|---:|---:|---:|---:|---:|---:|---|---|"]
    for key,score in stage["scores"].items():
        vals=[f"{score['pairs'][k]} ({pct(score['pairs'][k]/480)})" for k in ("D->0","B->8","8->B","0->D")]
        gaps=[k for k,v in stage["gates"][key].items() if not v]
        rows.append("| "+str(key)+" | "+pct(score["oof"])+" | "+pct(score["val"])+
            " | "+" | ".join(vals)+" | "+
            ("통과" if all(score["pairs"][k]<24 for k in ("D->0","B->8","8->B","0->D")) else "미달")+
            " | "+(", ".join(gaps) if gaps else "없음")+" |")
    return rows
def main():
    p=argparse.ArgumentParser()
    for x in ("s4","s5","logs","hp-index","combined-readme","master-log-index"):
        p.add_argument("--"+x,type=Path,required=True)
    a=p.parse_args()
    s4=json.loads(a.s4.read_text(encoding="utf-8"))
    s5=json.loads(a.s5.read_text(encoding="utf-8"))
    assert s4["stage"]=="S4_learning_rate" and s5["stage"]=="S5_early_stopping"
    lr=s4["selected_learning_rate"]
    es=s5["selected_early_stopping"]
    assert lr==s5["selected_learning_rate"] and str(es)==s5["selected"]
    assert s4["no_independent_test_used"] and s5["no_independent_test_used"]
    assert s4["same_official_and_handmade_sha_confirmed"] and s5["same_official_and_handmade_sha_confirmed"]
    assert all(s4["gates"][lr].values()) and all(s5["gates"][str(es)].values())
    score=s5["scores"][str(es)]
    lines=["# EXP-CMB-008 · 11 S1~S5 최종 HP 연구 결론","",
      "> **개발용 OOF/OfficialVal 기준 잠정 선택. Original Test·교수님 Test·신규 손글씨 480장 미사용. 최종 배포 체크포인트 미변경.**","",
      "## 단계별 가설 → 실측 → 결정 → 다음 질문","",
      "| 단계 | 비교와 선정 이유 | 결정 | 다음으로 넘어간 이유 |",
      "|---|---|---|---|",
      "| S1 너비 | (64,)/(100,)/(128,)/(256,)을 비교 | (256,)의 OOF 92.214%와 주요 오류 개선 | 표현 용량을 확인한 후 깊이 검증 |",
      "| S2 깊이 | (256,128)/(256,256)으로 모델 복잡도 증가 | 두 층에서 8→B 등의 오류 악화 → 단층 (256,) 유지 | 구조를 동결하고 L2 강도 확인 |",
      "| S3 L2 | alpha 1e-5/1e-4/1e-3/1e-2 | 강한 규제에서 D→0 악화 → alpha=1e-4 유지 | L2를 고정하고 최적화 속도 효과 분리 |",
      f"| S4 LR | 0.0005 / 0.001 / 0.002 | **{lr} 선택** ({s4['status']}) | LR 고정 후 Early Stopping 효과 검증 |",
      f"| S5 종료 | Early Stopping False vs True (Train 내부 10%) | **{es} 선택** ({s5['status']}) | HP 잠정 동결 |","",
      "## S4 Learning Rate 실측·선정",*table(s4),"",
      "## S5 Early Stopping 실측·선정",*table(s5),"",
      "## 현재 동결할 학습 조건","",
      "| 파라미터 | 선정값 |",
      "|---|---|",
      "| Train augmentation | Size 30% × Stroke 강도 28% (Official Train 일부 1:1 대체) |",
      "| Center | CoM |","| Hidden layers | (256,) |",
      "| Solver/activation | Adam / ReLU |",
      "| alpha | 0.0001 |",
      f"| learning_rate_init | **{lr}** |",
      f"| early_stopping | **{es}** |",
      "| max_iter | 200 |",
      f"| OOF Accuracy | **{pct(score['oof'])}** |",
      f"| OfficialVal Accuracy | **{pct(score['val'])}** |",
      f"| D→0 | {score['pairs']['D->0']}/480 = {pct(score['pairs']['D->0']/480)} |",
      f"| B→8 | {score['pairs']['B->8']}/480 = {pct(score['pairs']['B->8']/480)} |",
      f"| 8→B | {score['pairs']['8->B']}/480 = {pct(score['pairs']['8->B']/480)} |",
      f"| 0→D | {score['pairs']['0->D']}/480 = {pct(score['pairs']['0->D']/480)} |",""]
    target=all(score["pairs"][k]<24 for k in ("D->0","B->8","8->B","0->D"))
    lines+=[("**네 방향 각 5% 미만의 목표 달성**" if target else
        "**네 방향 각 5% 미만의 목표 미달.** 오분류 비증가와 작성자/Val/Flip 안전 Gate를 통과한 최선의 개발용 대안을 선택했다."),
      "**과적합 한계:** 보류 Writer 4명을 반복 탐색에 사용해 OOF 자체의 선택 편향 가능성은 남는다. ES 내부 10%의 검증은 Training Fold만 사용하며 독립 Test를 대체하지 않는다.","",
      "## 추적 가능한 원본","",
      "- [S1 너비](EXP-CMB-008_02_Stage1_Width_Results.md)",
      "- [S2 깊이](EXP-CMB-008_04_Stage2_Depth_Results.md)",
      "- [S3 L2](EXP-CMB-008_06_Stage3_Alpha_Results.md)",
      "- [S4 사전등록](EXP-CMB-008_07_Plan_S4_LearningRate.md) → [S4 실측](EXP-CMB-008_08_Stage4_LearningRate_Results.md)",
      "- [S5 사전등록](EXP-CMB-008_09_Plan_S5_EarlyStopping.md) → [S5 실측](EXP-CMB-008_10_Stage5_EarlyStopping_Results.md)",
      "- 원본 실측 CSV/JSON: ../../results/Hyperparameter_Optimization/S4_LearningRate/ 및 S5_EarlyStopping/",""]
    a.logs.mkdir(parents=True,exist_ok=True)
    (a.logs/"EXP-CMB-008_11_Final_HP_Decision.md").write_text("\n".join(lines),encoding="utf-8")
    for name,file in (("hp",a.hp_index),("combined",a.combined_readme),("master",a.master_log_index)):
        if name=="hp":
            body=f"**S1~S5 완료:** 단층 (256,), alpha=1e-4, LR **{lr}**, Early Stopping **{es}**, OOF **{pct(score['oof'])}**, OfficialVal **{pct(score['val'])}**. [S4 결과](EXP-CMB-008_08_Stage4_LearningRate_Results.md) → [S5 결과](EXP-CMB-008_10_Stage5_EarlyStopping_Results.md) → [최종 연구결정](EXP-CMB-008_11_Final_HP_Decision.md). 최종 Test·배포 미변경."
        elif name=="combined":
            body=f"**CMB-008 S1~S5 검증 완료:** (256,), alpha=1e-4, LR **{lr}**, Early Stopping **{es}**, OOF **{pct(score['oof'])}**, OfficialVal **{pct(score['val'])}**. [선정·탈락 근거](logs/하이퍼파라미터_최적화/EXP-CMB-008_11_Final_HP_Decision.md). 최종 Test·배포 미변경."
        else:
            body=f"**CMB-008 S1~S5 완료:** (256,), alpha=1e-4, LR **{lr}**, Early Stopping **{es}**, OOF **{pct(score['oof'])}**, OfficialVal **{pct(score['val'])}**. [S4 결과](하이퍼파라미터_최적화/EXP-CMB-008_08_Stage4_LearningRate_Results.md) → [S5 결과](하이퍼파라미터_최적화/EXP-CMB-008_10_Stage5_EarlyStopping_Results.md) → [최종 결정](하이퍼파라미터_최적화/EXP-CMB-008_11_Final_HP_Decision.md). 최종 Test·배포 미변경."
        file.write_text(with_banner(file.read_text(encoding="utf-8"),body),encoding="utf-8")
    print("FINAL_DECISION",json.dumps({"learning_rate":lr,"early_stopping":es,"oof":score["oof"],"official_val":score["val"],"four_directions_below_5pct":target},ensure_ascii=False),flush=True)
if __name__=="__main__":main()


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")